In [6]:
# Scenario: AI Research Assistant for a Corporate Innovation Team
# Imagine you’re part of a corporate innovation lab that constantly reviews new AI research papers to stay ahead of
# trends. The team struggles with long PDFs full of technical jargon, and they want a quick way to ask natural questions
# about the papers instead of reading them cover to cover.
# How the RAG Chatbot Fits In
# - Input Source: The team uploads a research paper (e.g., ai_research.pdf).
# - Chunking: The chatbot splits the paper into manageable sections so no detail is lost.
# - Embeddings + Vector DB: Each section is converted into embeddings and stored in Chroma, making the paper searchable by meaning rather than keywords.
# - Retriever: When someone asks, “What does this paper say about reinforcement learning?”, the retriever pulls the most relevant chunks.
# - LLM Response: The Hugging Face model (Flan-T5) generates a concise, human-readable answer using those chunks as context.
# - Chat Loop: The team can keep asking questions interactively, like a research assistant that knows the paper inside out.

# ================================
# AI Research Assistant (RAG + Gradio)
# ================================

# Install once if needed:
# pip install chromadb sentence-transformers pypdf transformers torch gradio

import os
import re
import torch
import chromadb
import gradio as gr
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM


# ========= CONFIG =========
PDF_PATH = "ai_research.pdf"  # Change path if needed
CHROMA_PATH = "./research_rag_db"
COLLECTION_NAME = "research_collection"


# ========= LOAD PDF =========
def load_pdf_text(path):
    reader = PdfReader(path)
    text = ""
    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text += page_text + "\n"
    return text


def clean_text(text):
    text = re.sub(r"\n{2,}", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    return text.strip()


# ========= CHUNKING =========
def chunk_text(text, chunk_size=600, overlap=100):
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        start += chunk_size - overlap

    return chunks


# ========= VECTOR DB =========
def create_collection():
    client = chromadb.PersistentClient(path=CHROMA_PATH)
    try:
        client.delete_collection(COLLECTION_NAME)
    except:
        pass
    return client.create_collection(name=COLLECTION_NAME)


def store_chunks(collection, chunks, embed_model):
    for i, chunk in enumerate(chunks):
        embedding = embed_model.encode(chunk).tolist()
        collection.add(
            ids=[str(i)],
            documents=[chunk],
            embeddings=[embedding]
        )


def retrieve(collection, embed_model, query, k=3):
    query_embedding = embed_model.encode(query).tolist()
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k
    )
    return results["documents"][0]


# ========= PROMPT =========
def build_prompt(context, query):
    return f"""
You are an AI research assistant.

Answer using ONLY the context below.
If the answer is not present, say "Not found in document".

Context:
{context}

Question:
{query}

Answer:
"""


# ========= GENERATION =========
def generate_answer(prompt):
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    )

    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=False
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)


def answer_question(query):
    if not query.strip():
        return "Please enter a question."

    docs = retrieve(collection, embedding_model, query)
    context = " ".join(docs)
    prompt = build_prompt(context, query)
    return generate_answer(prompt)


# ========= INITIALIZE SYSTEM =========
print("Loading PDF...")

if not os.path.exists(PDF_PATH):
    raise FileNotFoundError(f"PDF not found at {PDF_PATH}")

raw_text = load_pdf_text(PDF_PATH)
cleaned = clean_text(raw_text)
chunks = chunk_text(cleaned)

print("Total Chunks:", len(chunks))

print("Loading embedding model...")
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

print("Creating vector database...")
collection = create_collection()
store_chunks(collection, chunks, embedding_model)

print("Loading LLM...")
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")

print("System Ready.")


# ========= GRADIO UI =========
example_questions = [
    "What is Artificial Intelligence?",
    "Explain the transformer architecture.",
    "What does the paper say about reinforcement learning?",
    "What are Large Language Models?",
    "Summarize the main ideas of the paper."
]

demo = gr.Interface(
    fn=answer_question,
    inputs=gr.Textbox(label="Ask a research question"),
    outputs=gr.Textbox(label="Answer"),
    title="AI Research Assistant (RAG)",
    description="Ask questions about the uploaded research paper.",
    examples=example_questions
)

demo.launch()

Loading PDF...
Total Chunks: 6
Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Creating vector database...
Loading LLM...


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


System Ready.
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://3ec0abd3c559e3be32.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [2]:
# Scenario: Healthcare Policy Navigator
# Background:
# A large hospital network must comply with new government regulations related to
# telemedicine practices and patient data privacy. These regulations describe
# how patient data must be handled, what rights patients have, how cross-border
# telemedicine should work, and what penalties apply for violations.
#
# Challenge:
# The hospital compliance team must quickly understand the regulation document.
# Instead of manually reading hundreds of pages, the hospital deploys an AI
# Policy Navigator that can read the regulation PDF and answer questions.
#
# The system processes the regulation document, stores it in a vector database,
# retrieves relevant sections when a question is asked, and generates clear
# answers for doctors, administrators, and IT staff.

# Technologies Used:
# Python – main programming language
# PyPDF – extract text from regulation PDF
# Sentence Transformers – convert text into embeddings
# ChromaDB – store embeddings and enable semantic search
# HuggingFace Transformers – generate answers using FLAN-T5
# Gradio – create a simple web interface for asking questions

# ==========================================
# Healthcare Policy Navigator (Advanced UI)
# ==========================================

# pip install chromadb sentence-transformers pypdf transformers torch gradio

import os
import re
import torch
import chromadb
import gradio as gr

from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM


# ========= CONFIG =========
PDF_PATH = "National Telemedicine Privacy Act, 2026.pdf"
CHROMA_PATH = "./health_policy_db"
COLLECTION_NAME = "health_policy_collection"


# ========= LOAD PDF =========
def load_pdf_text(path):
    reader = PdfReader(path)
    text = ""
    for i, page in enumerate(reader.pages):
        page_text = page.extract_text()
        if page_text:
            text += f"\n\nPage {i+1}\n{page_text}"
    return text


def clean_text(text):
    text = text.replace("\xa0", " ")
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    return text.strip()


def chunk_text(text, size=700):
    return [text[i:i+size] for i in range(0, len(text), size)]


# ========= VECTOR DATABASE =========
def create_collection():
    client = chromadb.PersistentClient(path=CHROMA_PATH)
    try:
        client.delete_collection(COLLECTION_NAME)
    except:
        pass
    return client.create_collection(name=COLLECTION_NAME)


def store_chunks(collection, chunks, embed_model):
    for i, chunk in enumerate(chunks):
        embedding = embed_model.encode(chunk).tolist()
        collection.add(
            ids=[f"id{i}"],
            documents=[chunk],
            embeddings=[embedding]
        )


def retrieve_context(query):
    q_embedding = embed_model.encode(query).tolist()
    results = collection.query(
        query_embeddings=[q_embedding],
        n_results=3
    )
    docs = results["documents"][0]
    return docs


# ========= PROMPT =========
def build_prompt(context, question):
    return f"""
You are a healthcare compliance assistant.

Use ONLY the regulation context to answer.
If answer is not present, say "Not found in document".

Context:
{context}

Question:
{question}

Answer clearly so hospital staff can understand:
"""


# ========= GENERATE =========
def answer_question(question):

    if not question.strip():
        return "Please enter a question.", ""

    docs = retrieve_context(question)
    context = "\n\n".join(docs)

    prompt = build_prompt(context, question)

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    )

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=250
        )

    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return answer, context


# ========= INITIALIZATION =========
print("Loading regulation PDF")

if not os.path.exists(PDF_PATH):
    raise FileNotFoundError(f"PDF not found at {PDF_PATH}")

text = clean_text(load_pdf_text(PDF_PATH))
chunks = chunk_text(text)

print("Chunks created:", len(chunks))

print("Loading embedding model")
embed_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

print("Creating vector database")
collection = create_collection()
store_chunks(collection, chunks, embed_model)

print("Loading language model")
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")

print("System Ready")


# ========= GRADIO ADVANCED UI =========
with gr.Blocks(theme=gr.themes.Soft(), css="footer{display:none !important}") as demo:

    gr.Markdown("# 🏥 Healthcare Policy Navigator")
    gr.Markdown("Ask questions about telemedicine privacy and healthcare regulations.")

    with gr.Row():
        question_input = gr.Textbox(
            label="Ask a Question",
            placeholder="Example: What penalties apply for data misuse?",
            lines=2,
            scale=4
        )

    with gr.Row():
        ask_btn = gr.Button("Get Answer", variant="primary")
        clear_btn = gr.Button("Clear")

    with gr.Row():
        answer_output = gr.Textbox(
            label="📌 Final Answer",
            lines=8
        )

    with gr.Row():
        context_output = gr.Textbox(
            label="📄 Retrieved Regulation Sections",
            lines=12
        )

    ask_btn.click(
        fn=answer_question,
        inputs=question_input,
        outputs=[answer_output, context_output]
    )

    question_input.submit(
        fn=answer_question,
        inputs=question_input,
        outputs=[answer_output, context_output]
    )

    clear_btn.click(
        fn=lambda: ("", "", ""),
        inputs=[],
        outputs=[question_input, answer_output, context_output]
    )

demo.launch()


# What compliance risks does this regulation create for hospitals using telemedicine platforms?

# What rights do patients have under this telemedicine privacy regulation?

# What penalties or legal consequences are mentioned for violating patient data privacy rules?

# What operational steps must a hospital take to comply with this telemedicine regulation?

Loading regulation PDF
Chunks created: 3
Loading embedding model


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

System Ready


/tmp/ipykernel_5020/525506669.py:173: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), css="footer{display:none !important}") as demo:
/tmp/ipykernel_5020/525506669.py:173: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), css="footer{display:none !important}") as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2ad71e80103641ad84.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
# Scenario: University Library Assistant
# A large university library has thousands of digitized textbooks, research papers, and course notes.
# Students often struggle to find specific explanations or summaries when preparing for exams.
# This project builds a simple RAG chatbot without LangChain.
# It reads a textbook PDF, splits it into chunks, creates embeddings,
# stores them in ChromaDB, retrieves relevant sections, and uses Flan-T5
# to answer student questions in clear and simple language.

# Install required libraries before running:
# pip install chromadb sentence-transformers pypdf transformers torch

import os
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import chromadb
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch


PDF_PATH = "Introduction_to_Data_Science.pdf"
CHROMA_PATH = "./library_rag_db"
COLLECTION_NAME = "university_library_collection"


def load_pdf_text(pdf_path):
    reader = PdfReader(pdf_path)
    full_text = ""

    for page_number, page in enumerate(reader.pages, start=1):
        page_text = page.extract_text()
        if page_text:
            full_text += f"\n\n[Page {page_number}]\n{page_text}"

    return full_text


def chunk_text(text, chunk_size=500, overlap=50):
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()

        if chunk:
            chunks.append(chunk)

        start += chunk_size - overlap

    return chunks


def create_collection():
    client = chromadb.PersistentClient(path=CHROMA_PATH)

    try:
        client.delete_collection(COLLECTION_NAME)
    except Exception:
        pass

    collection = client.create_collection(name=COLLECTION_NAME)
    return client, collection


def store_chunks(collection, chunks, embedding_model):
    for i, chunk in enumerate(chunks):
        embedding = embedding_model.encode(chunk).tolist()

        collection.add(
            documents=[chunk],
            embeddings=[embedding],
            ids=[f"chunk_{i}"]
        )


def retrieve(query, collection, embedding_model, k=3):
    query_embedding = embedding_model.encode(query).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k
    )

    return results["documents"][0]


def build_prompt(context, query):
    prompt = f"""
You are a university library study assistant.

Answer the student's question using ONLY the context below.
If the answer is not present, say: Not found in document.
Explain in simple, clear, student-friendly language.
Keep the answer concise and accurate.

Context:
{context}

Question:
{query}

Answer:
"""
    return prompt.strip()


def generate_answer(tokenizer, model, prompt, max_new_tokens=180):
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    )

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False
        )

    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return answer


def answer_question(query, collection, embedding_model, tokenizer, model):
    context_docs = retrieve(query, collection, embedding_model, k=3)
    context = "\n\n".join(context_docs)

    prompt = build_prompt(context, query)
    answer = generate_answer(tokenizer, model, prompt)

    return answer, context_docs


def main():
    if not os.path.exists(PDF_PATH):
        print("PDF file not found. Check PDF_PATH.")
        return

    print("Loading PDF document...")
    text = load_pdf_text(PDF_PATH)

    print("Document loaded")
    print("Total characters:", len(text))

    print("\nSplitting document into chunks...")
    chunks = chunk_text(text, chunk_size=500, overlap=50)

    print("Total chunks created:", len(chunks))

    print("\nLoading embedding model...")
    embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
    print("Embedding model loaded")

    print("\nCreating vector database...")
    _, collection = create_collection()
    print("New vector collection created")

    print("\nCreating embeddings and storing in ChromaDB...")
    store_chunks(collection, chunks, embedding_model)
    print("All chunks stored successfully")

    print("\nLoading LLM...")
    tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
    model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")
    print("LLM loaded successfully")

    print("\n==============================")
    print("University Library RAG Chatbot Ready")
    print("Type 'exit' to stop")
    print("==============================\n")

    print("Example Questions:")
    print("1. What is Data Science?")
    print("2. Explain the difference between supervised and unsupervised learning.")
    print("3. What are the steps in the Data Science workflow?")
    print("4. What tools are used in data science?")
    print("5. Summarize the main concepts of the document.\n")

    while True:
        try:
            question = input("Ask a question: ").strip()
        except KeyboardInterrupt:
            print("\nGoodbye!")
            break

        if question.lower() == "exit":
            print("Goodbye!")
            break

        if not question:
            print("Please enter a valid question.\n")
            continue

        answer, context_docs = answer_question(
            question,
            collection,
            embedding_model,
            tokenizer,
            model
        )

        print("\nTop Retrieved Chunks:\n")
        for i, doc in enumerate(context_docs, start=1):
            print(f"Chunk {i}:")
            print(doc[:400])
            print("-" * 60)

        print("\nAnswer:\n")
        print(answer)
        print("\n" + "-" * 60 + "\n")


if __name__ == "__main__":
    main()

Loading PDF document...
Document loaded
Total characters: 3460

Splitting document into chunks...
Total chunks created: 8

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded

Creating vector database...
New vector collection created

Creating embeddings and storing in ChromaDB...
All chunks stored successfully

Loading LLM...


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


LLM loaded successfully

University Library RAG Chatbot Ready
Type 'exit' to stop

Example Questions:
1. What is Data Science?
2. Explain the difference between supervised and unsupervised learning.
3. What are the steps in the Data Science workflow?
4. What tools are used in data science?
5. Summarize the main concepts of the document.


Top Retrieved Chunks:

Chunk 1:
[Page 1]
Introduction to Data Science
Data Science is an interdisciplinary field that combines statistics, computer science, and domain
knowledge to extract meaningful insights from data. In today's digital world, organizations generate
massive amounts of data from websites, mobile applications, sensors, financial systems, and social
media platforms. Data scientists analyze this data to help busin
------------------------------------------------------------
Chunk 2:
s by enabling organizations to make data-driven decisions.
With the rapid growth of data, the demand for skilled data scientists continues to rise.
Understa

In [ ]:
# Scenario: AI Research Assistant for a Corporate Innovation Team
# Imagine you’re part of a corporate innovation lab that constantly reviews new AI research papers to stay ahead of
# trends. The team struggles with long PDFs full of technical jargon, and they want a quick way to ask natural questions
# about the papers instead of reading them cover to cover.
# How the RAG Chatbot Fits In
# - Input Source: The team uploads a research paper (e.g., ai_research.pdf).
# - Chunking: The chatbot splits the paper into manageable sections so no detail is lost.
# - Embeddings + Vector DB: Each section is converted into embeddings and stored in Chroma, making the paper searchable by meaning rather than keywords.
# - Retriever: When someone asks, “What does this paper say about reinforcement learning?”, the retriever pulls the most relevant chunks.
# - LLM Response: The Hugging Face model (Flan-T5) generates a concise, human-readable answer using those chunks as context.
# - Chat Loop: The team can keep asking questions interactively, like a research assistant that knows the paper inside out.

# ==========================================================
# SIMPLE RAG CHATBOT (NO LANGCHAIN) — FULLY ANNOTATED
# ==========================================================

# ----------------------------------------------------------
# STEP 0 — Install Required Libraries
# ----------------------------------------------------------
# chromadb → vector database
# sentence-transformers → embedding model
# pypdf → reading PDF files
# transformers → running the LLM




# ----------------------------------------------------------
# STEP 1 — Import Libraries
# ----------------------------------------------------------

import os

# Library for reading PDF documents
from pypdf import PdfReader

# Embedding model
from sentence_transformers import SentenceTransformer

# Vector database
import chromadb

# HuggingFace model pipeline
from transformers import pipeline


# ----------------------------------------------------------
# STEP 2 — Load the PDF Document
# ----------------------------------------------------------
# This document acts as the knowledge source for RAG

print("Loading PDF document...")

reader = PdfReader(r"/content/ai_research.pdf")

text = ""

# Extract text from every page
for page in reader.pages:
    text += page.extract_text()

print("Document Loaded")
print("Total Characters:", len(text))

# Preview some text
print("\nPreview:\n")
print(text[:500])


# ----------------------------------------------------------
# STEP 3 — Chunk the Document
# ----------------------------------------------------------
# LLMs work better with smaller text segments
# so we split the document into chunks

print("\nSplitting document into chunks...")

def chunk_text(text, chunk_size=500, overlap=50):
    """
    Split text into overlapping chunks

    chunk_size = max characters per chunk
    overlap = shared characters between chunks
    """

    chunks = []
    start = 0

    while start < len(text):

        end = start + chunk_size
        chunk = text[start:end]

        chunks.append(chunk)

        start += chunk_size - overlap

    return chunks


chunks = chunk_text(text)

print("Total Chunks Created:", len(chunks))

print("\nExample Chunk:\n")
print(chunks[0])


# ----------------------------------------------------------
# STEP 4 — Create Embeddings
# ----------------------------------------------------------
# Convert each chunk into a numerical vector
# These vectors allow semantic similarity search

print("\nLoading embedding model...")

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded")


# ----------------------------------------------------------
# STEP 5 — Create Vector Database
# ----------------------------------------------------------
# ChromaDB stores embeddings and documents

client = chromadb.Client()

# Delete collection if it exists
try:
    client.delete_collection("pdf_collection")
    print("Old collection deleted")
except:
    pass


collection = client.create_collection("pdf_collection")

print("New vector collection created")


# ----------------------------------------------------------
# STEP 6 — Store Chunks in Vector DB
# ----------------------------------------------------------

print("\nCreating embeddings and storing in ChromaDB...")

for i, chunk in enumerate(chunks):

    # Create embedding vector
    embedding = embedding_model.encode(chunk).tolist()

    # Store in vector database
    collection.add(
        documents=[chunk],
        embeddings=[embedding],
        ids=[str(i)]
    )

print("All chunks stored successfully")


# ----------------------------------------------------------
# STEP 7 — Retriever Function
# ----------------------------------------------------------
# Converts user question → embedding
# Finds most similar document chunks

def retrieve(query, k=3):

    # Convert query to embedding
    query_embedding = embedding_model.encode(query).tolist()

    # Perform similarity search
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k
    )

    return results["documents"][0]


# ----------------------------------------------------------
# STEP 8 — Load the LLM
# ----------------------------------------------------------
# We use Flan-T5 for answer generation

print("\nLoading LLM...")

qa_pipeline = pipeline(
    "text-generation",
    model="google/flan-t5-base"
)

print("LLM loaded successfully")


# ----------------------------------------------------------
# STEP 9 — Question Answering Function
# ----------------------------------------------------------
# RAG Workflow:
#
# Question
#   ↓
# Retrieve Relevant Chunks
#   ↓
# Send Context + Question to LLM
#   ↓
# Generate Answer

def answer_question(query):

    # Retrieve relevant context
    context_docs = retrieve(query)

    context = " ".join(context_docs)

    prompt = f"""
You are an AI assistant.

Answer the question using ONLY the context below.
If the answer is not present, say "Not found in document".

Context:
{context}

Question:
{query}

Answer:
"""

    response = qa_pipeline(
        prompt,
        max_length=256,
        temperature=0.5
    )

    return response[0]["generated_text"]


# ----------------------------------------------------------
# STEP 10 — Chat Loop
# ----------------------------------------------------------
# Interactive question-answering interface

print("\n==============================")
print("RAG Chatbot Ready")
print("Type 'exit' to stop")
print("==============================\n")

while True:

    question = input("Ask a question: ")

    if question.lower() == "exit":
        print("Goodbye!")
        break

    answer = answer_question(question)

    print("\nAnswer:\n", answer)
    print("\n" + "-"*60 + "\n")

In [6]:
# Scenario: "The Environmental Policy Compliance Assistant"
# Background
# You are part of the Sustainability & Environmental Compliance Team at a global manufacturing company.
# New government regulations on carbon emissions, waste disposal, and renewable energy adoption have just been released.
# The company must ensure compliance to avoid fines and reputational damage.

# Challenge
# The company uploads a PDF of the environmental regulation into the Compliance Assistant (Gradio + Chroma + LLM app).
# Your task is to:
# - Process the regulation document so the assistant can store and understand it.
# - Ask compliance-related questions about emission limits, waste management rules, and renewable energy targets.
# - Generate clear, actionable answers that can guide engineers, sustainability officers, and executives.

# Roles
# - Learner (You): Environmental compliance officer using the assistant.
# - Assistant (The RAG App): Provides answers strictly based on uploaded environmental regulations.
# - Stakeholders: Plant managers, sustainability officers, and executives who need concise compliance guidance.

# 🔄 Flow of the Scenario
# - Upload Environmental Regulation PDF
# Example: “National Carbon Emissions Act 2026”.
# - System Processes Document
# - Splits into chunks.
# - Embeds into vector database.
# - Stores for retrieval.
# - Ask Questions
# - “What is the maximum carbon emission allowed per factory per year?”
# - “What penalties apply if hazardous waste is not disposed of properly?”
# - “What renewable energy targets must we meet by 2030?”
# - Assistant Responds
# - Retrieves relevant chunks.
# - Generates compliance-focused answers.
# - Provides short, clear guidance.
# - Outcome
# - Learners practice extracting environmental obligations.
# - Managers receive summarized compliance insights.
# - Executives gain confidence in sustainability strategy alignment.

# 🎯 Training Objective
# This scenario helps learners:
# - Understand how RAG systems can support environmental compliance.
# - Practice formulating precise queries to extract obligations.
# - Experience how AI can simplify complex environmental regulations into actionable steps.

# 👉 Would you like me to also draft a sample regulation PDF text (like the healthcare one I created earlier) for this environmental context, so you can upload it into your assistant and simulate queries?



#  Environmental Policy Compliance Assistant
# Simple + Clean Gradio Version


!pip install chromadb sentence-transformers pypdf transformers torch gradio

import os
import re
import torch
import chromadb
import gradio as gr
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM


# ================= CONFIG =================

PDF_PATH = "National Carbon Emissions & Sustainability Act, 2026.pdf"
CHROMA_PATH = "./environment_policy_db"
COLLECTION_NAME = "environment_collection"


# ================= PDF PROCESSING =================

def load_pdf_text(path):
    reader = PdfReader(path)
    text = ""
    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text += page_text + "\n"
    return text


def clean_text(text):
    text = text.replace("\xa0", " ")
    text = re.sub(r"\n{2,}", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    return text.strip()


def chunk_text(text, size=800, overlap=150):
    chunks = []
    start = 0
    while start < len(text):
        end = start + size
        chunks.append(text[start:end])
        start += size - overlap
    return chunks


# ================= VECTOR DATABASE =================

def create_collection():
    client = chromadb.PersistentClient(path=CHROMA_PATH)
    try:
        client.delete_collection(COLLECTION_NAME)
    except:
        pass
    return client.create_collection(name=COLLECTION_NAME)


def store_chunks(collection, chunks):
    for i, chunk in enumerate(chunks):
        embedding = embed_model.encode(chunk).tolist()
        collection.add(
            ids=[f"id{i}"],
            documents=[chunk],
            embeddings=[embedding]
        )


def retrieve_context(query):
    query_embedding = embed_model.encode(query).tolist()
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=5   # increased for better retrieval
    )
    return "\n\n".join(results["documents"][0])


# ================= PROMPT =================

def build_prompt(context, question):
    return f"""
You are an Environmental Compliance Assistant.

Use ONLY the regulation context to answer.

Even if wording differs, identify:
- Emission limits
- Waste rules
- Renewable targets
- Penalties
- Deadlines

If clearly not mentioned, say:
Not found in document.

Context:
{context}

Question:
{question}

Answer:
"""


# ================= ANSWER FUNCTION =================

def generate_answer(question, history):

    if not question.strip():
        return history

    context = retrieve_context(question)
    prompt = build_prompt(context, question)

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    )

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False
        )

    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

    history.append((question, answer))
    return history


# ================= INITIALIZATION =================

print("Loading PDF...")

if not os.path.exists(PDF_PATH):
    raise FileNotFoundError(f"PDF not found at {PDF_PATH}")

text = clean_text(load_pdf_text(PDF_PATH))
chunks = chunk_text(text)

print("Chunks created:", len(chunks))

print("Loading embedding model...")
embed_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

print("Creating vector DB...")
collection = create_collection()
store_chunks(collection, chunks)

print("Loading language model...")
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")

print("System Ready ✅")


# ================= SIMPLE GRADIO UI =================

with gr.Blocks(theme=gr.themes.Soft()) as demo:

    gr.Markdown("## 🌱 Environmental Policy Compliance Assistant")
    gr.Markdown("Ask questions about emissions, waste rules, penalties, or renewable targets.")

    chatbot = gr.Chatbot(height=450)

    msg = gr.Textbox(
        placeholder="Example: What is the emission limit per factory?",
        label="Ask your question"
    )

    clear = gr.Button("Clear Chat")

    msg.submit(generate_answer, [msg, chatbot], chatbot)
    clear.click(lambda: [], None, chatbot)

demo.launch()

Loading PDF...
Chunks created: 4
Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Creating vector DB...
Loading language model...


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
/tmp/ipykernel_5020/2038445274.py:170: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


System Ready ✅


/tmp/ipykernel_5020/2038445274.py:175: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(height=450)
/tmp/ipykernel_5020/2038445274.py:175: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(height=450)


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ab567fdfcc30714f8f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [8]:
# pip install chromadb sentence-transformers pypdf transformers torch gradio

import os
import re
import torch
import gradio as gr
import chromadb
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM


PDF_PATH = "data_privacy_regulation.pdf"

CHROMA_PATH = "./legal_rag_db"
COLLECTION_NAME = "legal_compliance_collection"


def load_pdf_text(pdf_path):
    reader = PdfReader(pdf_path)
    text = ""

    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text += page_text + "\n"

    return text


def clean_text(text):
    text = re.sub(r"\n{2,}", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    return text.strip()


def chunk_text(text, chunk_size=700, overlap=120):
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        start += chunk_size - overlap

    return chunks


def create_collection():
    client = chromadb.PersistentClient(path=CHROMA_PATH)

    try:
        client.delete_collection(COLLECTION_NAME)
    except:
        pass

    collection = client.create_collection(name=COLLECTION_NAME)

    return collection


def store_chunks(collection, chunks, embedding_model):
    for i, chunk in enumerate(chunks):

        embedding = embedding_model.encode(chunk).tolist()

        collection.add(
            ids=[str(i)],
            documents=[chunk],
            embeddings=[embedding]
        )


def retrieve(collection, embedding_model, query, k=3):

    query_embedding = embedding_model.encode(query).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k
    )

    return results["documents"][0]


def build_prompt(context, query):

    prompt = f"""
You are a legal research assistant.

Answer using ONLY the context.

Context:
{context}

Question:
{query}

Answer:
"""

    return prompt


def generate_answer(prompt):

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    )

    outputs = model.generate(
        **inputs,
        max_new_tokens=200
    )

    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return answer


def answer_question(query):

    docs = retrieve(collection, embedding_model, query)

    context = " ".join(docs)

    prompt = build_prompt(context, query)

    answer = generate_answer(prompt)

    return answer


print("Loading PDF...")

text = load_pdf_text(PDF_PATH)

text = clean_text(text)

chunks = chunk_text(text)

print("Chunks:", len(chunks))


print("Loading embedding model...")

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)


print("Creating vector DB...")

collection = create_collection()

store_chunks(collection, chunks, embedding_model)


print("Loading LLM...")

tokenizer = AutoTokenizer.from_pretrained(
    "google/flan-t5-base"
)

model = AutoModelForSeq2SeqLM.from_pretrained(
    "google/flan-t5-base"
)


print("System ready")


def gradio_chat(question):
    return answer_question(question)


demo = gr.Interface(
    fn=gradio_chat,
    inputs=gr.Textbox(label="Ask legal question"),
    outputs=gr.Textbox(label="Answer"),
    title="Legal Research Assistant"
)

demo.launch()

Loading PDF...
Chunks: 5
Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Creating vector DB...
Loading LLM...


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


System ready
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://32660bb03b4ae8c50d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [4]:
 # Scenario: Legal Research Assistant for a Corporate Compliance Team
# Context
# A corporate compliance department constantly reviews lengthy legal documents, regulatory filings, and policy updates. These documents are dense, full of
# legal terminology, and often hundreds of pages long. The team struggles to quickly extract relevant clauses or understand implications without spending hours reading.
# How the RAG Chatbot Fits In
# - Input Source: The team uploads a legal document (e.g., data_privacy_regulation.pdf).
# - Chunking: The chatbot splits the document into sections (clauses, articles, sub-sections) so no detail is overlooked.
# - Embeddings + Vector DB: Each section is converted into embeddings and stored in Chroma, enabling semantic search rather than keyword-only lookup.
# - Retriever: When someone asks, “What does this regulation say about cross-border data transfers?”, the retriever surfaces the most relevant clauses.
# - LLM Response: A Hugging Face model (e.g., Flan-T5) generates a concise, plain-language summary of those clauses, stripping away heavy legal jargon.
# - Chat Loop: The compliance team can continue asking questions interactively, like “Does this regulation conflict with GDPR?” or “What penalties are mentioned
#  for non-compliance?”.
# Outcome
# The chatbot acts as a legal research assistant, helping the compliance team quickly interpret complex documents, identify risks, and prepare summaries for executives
#  without needing to manually parse every page.

# Scenario: Legal Research Assistant for a Corporate Compliance Team
# This project builds a simple RAG chatbot without LangChain.
# It reads a legal PDF, splits it into meaningful sections,
# creates embeddings, stores them in ChromaDB, retrieves relevant parts,
# and uses a Hugging Face model to answer questions in plain English.

# Install required libraries before running:

!pip install chromadb sentence-transformers pypdf transformers torch gradio

import os
import re
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import chromadb
from transformers import pipeline
import gradio as gr


PDF_PATH = "data_privacy_regulation.pdf"

CHROMA_PATH = "./legal_rag_db"
COLLECTION_NAME = "legal_compliance_collection"


def load_pdf_text(pdf_path):
    reader = PdfReader(pdf_path)
    full_text = ""

    for page_number, page in enumerate(reader.pages, start=1):
        page_text = page.extract_text()
        if page_text:
            full_text += f"\n\n[Page {page_number}]\n{page_text}"

    return full_text


def clean_text(text):
    text = text.replace("\u25a0", "-")
    text = re.sub(r"\r", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    return text.strip()


def split_text(text, chunk_size=700):
    chunks = []

    for i in range(0, len(text), chunk_size):
        chunk = text[i:i+chunk_size]
        chunks.append(chunk)

    return chunks


def create_collection():

    client = chromadb.PersistentClient(path=CHROMA_PATH)

    try:
        client.delete_collection(COLLECTION_NAME)
    except:
        pass

    collection = client.create_collection(name=COLLECTION_NAME)

    return collection


def store_chunks(collection, chunks, embedding_model):

    for i, chunk in enumerate(chunks):

        embedding = embedding_model.encode(chunk).tolist()

        collection.add(
            ids=[f"id_{i}"],
            documents=[chunk],
            embeddings=[embedding]
        )


def retrieve_chunks(collection, embedding_model, query):

    query_embedding = embedding_model.encode(query).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=3
    )

    return results["documents"][0]


def answer_query(query):

    docs = retrieve_chunks(collection, embedding_model, query)

    context = "\n".join(docs)

    prompt = f"""
Use the context to answer the question.

Context:
{context}

Question:
{query}

Answer:
"""

    response = qa_pipeline(prompt, max_length=200)

    return response[0]["generated_text"]


print("Loading PDF")

text = load_pdf_text(PDF_PATH)

text = clean_text(text)

chunks = split_text(text)

print("Chunks:", len(chunks))


print("Loading Embedding Model")

embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")


print("Creating Vector Database")

collection = create_collection()

store_chunks(collection, chunks, embedding_model)


print("Loading LLM")

qa_pipeline = pipeline(
    "text-generation",
    model="google/flan-t5-base"
)


def chat(question):

    answer = answer_query(question)

    return answer


demo = gr.Interface(
    fn=chat,
    inputs="text",
    outputs="text",
    title="Legal Research Assistant"
)

demo.launch()

# Basic Test Questions

# What does this regulation say about data privacy?

# What are the responsibilities of organizations handling personal data?

# Does the document mention data protection requirements?

# What is the definition of personal data in the regulation?

# What obligations do companies have under this regulation?

# Intermediate Questions

# What rules apply to cross-border data transfers?

# What are the security requirements for protecting user data?

# What happens if an organization fails to comply with the regulation?

# What rights do individuals have regarding their personal data?

# Are there any data breach notification requirements?

# Advanced / Real Compliance Questions

# What penalties or fines are mentioned for non-compliance?

# What are the data retention rules described in the document?

# Does the regulation describe consent requirements for data collection?

# What safeguards must companies implement to protect sensitive data?

# Does this regulation mention third-party data sharing rules?

# Edge Case Questions (RAG test)

# Ye important hai kyunki RAG ka real test hota hai.

# Does this document mention GDPR specifically?

# What does the regulation say about AI systems using personal data?

# Does the regulation specify deadlines for compliance?

# What reporting requirements are mentioned?

Loading PDF
Chunks: 5
Loading Embedding Model


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Creating Vector Database
Loading LLM


model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

...

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLlamaForCausalLM', 'DogeForCausalLM', 'Dots1ForCausalLM', 'ElectraForCausalLM', 'Emu3ForCausalLM', 'ErnieForCausalLM', 'Ernie4_5ForCausalLM', 'Ernie4_5_MoeForCausalLM', 'Exaone4ForCausalLM', 'FalconForCausalLM', 'FalconH1ForCausalLM', 'FalconMambaForCausa

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d96b74113ee37c0ec1.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
